In [1]:
import pandas as pd
import numpy as np
import requests
import streamlit as st
import sqlite3

In [10]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO

def scrape_cbb_data(season=2026, table_type="school_advanced"):
    """
    Valid table_types:
    - 'school_basic'
    - 'school_advanced'
    - 'opponent_basic'
    - 'opponent_advanced'
    """
    
    # 1. Map the table type to the exact URL configuration and HTML element ID
    config = {
        "school_basic": {
            "url_suffix": "school-stats.html",
            "table_id": "basic_school_stats"
        },
        "school_advanced": {
            "url_suffix": "advanced-school-stats.html",
            "table_id": "adv_school_stats"
        },
        "opponent_basic": {
            "url_suffix": "opponent-stats.html",
            "table_id": "basic_opp_stats"
        },
        "opponent_advanced": {
            "url_suffix": "advanced-opponent-stats.html",
            "table_id": "adv_opp_stats"
        }
    }
    
    if table_type not in config:
        raise ValueError(f"Invalid table_type. Choose from: {list(config.keys())}")
        
    # 2. Build the URL dynamically
    suffix = config[table_type]["url_suffix"]
    target_table_id = config[table_type]["table_id"]
    url = f"https://www.sports-reference.com/cbb/seasons/men/{season}-{suffix}"
    
    # 3. Fetch the web page
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print(f"Failed to fetch data. Status code: {response.status_code}")
        return None
        
    # 4. Parse HTML and find the specific table ID
    soup = BeautifulSoup(response.content, 'html.parser')
    table = soup.find('table', {'id': target_table_id})
    
    if table is None:
        print(f"Error: Could not find table with ID '{target_table_id}' on the page.")
        return None
        
    # 5. Read into Pandas
    df = pd.read_html(StringIO(str(table)))[0]
    
    # 6. Clean Multi-index headers
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ['_'.join(col).strip() for col in df.columns.values]
        
    # 7. Remove repeated header rows throughout the table
    school_col = [c for c in df.columns if 'School' in c][0]
    df = df[df[school_col] != 'School']
    df = df.reset_index(drop=True)
    
    return df

# --- HOW TO RUN IT FOR ALL 4 TABLES ---

tables_to_get = ["school_basic", "school_advanced", "opponent_basic", "opponent_advanced"]

for t_type in tables_to_get:
    print(f"Fetching {t_type} data...")
    
    # Scrape the data
    data = scrape_cbb_data(season=2026, table_type=t_type)
    
    if data is not None:
        # Generate a distinct filename for each table
        filename = f"cbb_{t_type}_2026.csv"
        data.to_csv(filename, index=False)
        print(f"--> Saved {data.shape[0]} teams to {filename}")
        
    # CRITICAL: Sleep 4 seconds between tables to respect the 20 requests per minute rule!
    print("Sleeping 4 seconds to respect rate limits...")
    time.sleep(4)

print("\nAll downloads complete! Check your folder for the 4 CSV files.")

Fetching school_basic data...
--> Saved 383 teams to cbb_school_basic_2026.csv
Sleeping 4 seconds to respect rate limits...
Fetching school_advanced data...
--> Saved 383 teams to cbb_school_advanced_2026.csv
Sleeping 4 seconds to respect rate limits...
Fetching opponent_basic data...
--> Saved 383 teams to cbb_opponent_basic_2026.csv
Sleeping 4 seconds to respect rate limits...
Fetching opponent_advanced data...
--> Saved 383 teams to cbb_opponent_advanced_2026.csv
Sleeping 4 seconds to respect rate limits...

All downloads complete! Check your folder for the 4 CSV files.
